In [9]:
import dlt
from dlt.sources.rest_api import rest_api_resources
from dlt.sources.rest_api.typing import RESTAPIConfig


# if no argument is provided, `access_token` is read from `.dlt/secrets.toml`
@dlt.source
def taxi_pipeline_rest_api_source():
    """Define dlt resources from REST API endpoints."""
    config: RESTAPIConfig = {
        "client": {
            # TODO set base URL for the REST API
            "base_url": "https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api",
            # TODO configure the right authentication method or remove
            # "auth": {"type": "bearer", "token": access_token},
        },
        "resources": [
            # TODO define resources per endpoint
            {"name": "nyc_taxitrips",
            "endpoint": {
                "path": "search.json",
                "params": {
                        "limit": 100,
                    },
                "data_selector": "docs",
                "paginator": {
                        "type": "offset",
                        "limit": 100,
                        "offset_param": "offset",
                        "limit_param": "limit",
                        "total_path": "numFound",
                    },
            }}
            
        ],
        # set `resource_defaults` to apply configuration to all endpoints
        "resource_defaults": {
            "primary_key": "key",
            "write_disposition": "replace",
        },
    }

    yield from rest_api_resources(config)


pipeline = dlt.pipeline(
    pipeline_name='taxi_pipeline_pipeline',
    destination='duckdb',
    # `refresh="drop_sources"` ensures the data and the state is cleaned
    # on each `pipeline.run()`; remove the argument once you have a
    # working pipeline.
    refresh="drop_sources",
    # show basic progress of resources extracted, normalized files and load-jobs on stdout
    progress="log",
)

In [10]:
extract_info = pipeline.extract(taxi_pipeline_rest_api_source)

-------------------- Extract taxi_pipeline_rest_api_source ---------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 207.20 MB (58.20%) | CPU usage: 0.00%

-------------------- Extract taxi_pipeline_rest_api_source ---------------------
Resources: 0/1 (0.0%) | Time: 2.86s | Rate: 0.00/s
nyc_taxitrips: 0  | Time: 0.00s | Rate: 0.00/s
Memory usage: 209.00 MB (58.10%) | CPU usage: 0.00%

-------------------- Extract taxi_pipeline_rest_api_source ---------------------
Resources: 1/1 (100.0%) | Time: 2.88s | Rate: 0.35/s
nyc_taxitrips: 0  | Time: 0.02s | Rate: 0.00/s
Memory usage: 209.00 MB (58.10%) | CPU usage: 0.00%

-------------------- Extract taxi_pipeline_rest_api_source ---------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 209.03 MB (58.20%) | CPU usage: 0.00%

-------------------- Extract taxi_pipeline_rest_api_source ---------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
_dlt_pipeline_state: 1  | Time: 

In [11]:
load_id = extract_info.loads_ids[-1]
m = extract_info.metrics[load_id][0]

In [12]:
print("Resources:", list(m["resource_metrics"].keys()))
print("Tables:", list(m["table_metrics"].keys()))
print("Load ID:", load_id)
print()

for resource, rm in m["resource_metrics"].items():
    print(f"Resource: {resource}")
    print(f"rows extracted: {rm.items_count}")
    print()

Resources: []
Tables: []
Load ID: 1772309834.0243418

